# CNN channel ablation: 128 vs 64
Train hai cấu hình P=128, C=128 và P=128, C=64 từ scratch. Cả hai giữ k=5, residual on, Top-K, loss và cùng seed dữ liệu. Kết quả chính là checkpoint epoch 10, không chọn epoch theo test AUC.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path
import torch
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
OUTPUT_ROOT = PROJECT_ROOT / 'code' / 'results'
RUN_TAG = 'cnn_k5_p128_c64_vs_c128_s234'
SEED = 234
DATA_SEED = 234
EPOCH = 10
CHANNELS_TO_RUN = (128, 64)

if not torch.cuda.is_available():
    raise RuntimeError('Hãy chọn GPU runtime trên Colab.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    archives = [PROJECT_ROOT / name for name in ('UCFClipFeatures.zip', 'UCFClipFeatures.tar', 'UCFClipFeatures.tar.gz')]
    archive = next((path for path in archives if path.is_file()), None)
    if archive is None:
        raise FileNotFoundError('Không tìm thấy UCFClipFeatures hoặc file nén trên Drive.')
    local_archive = Path('/content') / archive.name
    if not local_archive.is_file() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    command = ['unzip', '-q', str(local_archive), '-d', '/content'] if archive.suffix == '.zip' else ['tar', '-xf', str(local_archive), '-C', '/content']
    subprocess.run(command, check=True)
required = (SRC_DIR / 'model_cnn_vadclip.py', SRC_DIR / 'ucf_train_cnn_vadclip.py',
            SRC_DIR / 'ucf_evaluate_cnn_vadclip.py', SRC_DIR / 'ucf_cnn_common.py',
            LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
            LIST_DIR / 'gt_ucf.npy', LIST_DIR / 'gt_segment_ucf.npy', LIST_DIR / 'gt_label_ucf.npy', FEATURE_ROOT)
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
print('GPU:', torch.cuda.get_device_name(0))
print('Run tag:', RUN_TAG)


## Kiểm tra nhanh
C=64 có Conv đầu [64,128,5] và skip projection 128→64; C=128 giữ nguyên kiến trúc cũ.

In [ ]:
sys.path.insert(0, str(SRC_DIR))
from model_cnn_vadclip import TemporalCNNClassifier
for channels in CHANNELS_TO_RUN:
    cnn = TemporalCNNClassifier(projection_width=128, hidden_width=channels, kernel_size=5, use_residual=True)
    assert tuple(cnn.blocks[0].conv1.weight.shape) == (channels, 128, 5)
    assert tuple(cnn.blocks[1].conv1.weight.shape) == (channels, channels, 5)
    scores = cnn(torch.randn(2, 20, 512), torch.tensor([13, 20]))
    assert scores.shape == (2, 20, 1) and torch.isfinite(scores).all()
    assert torch.equal(scores[0, 13:], torch.zeros_like(scores[0, 13:]))
    scores[:, :13].sum().backward()
    assert cnn.blocks[0].conv1.weight.grad is not None
    if channels == 64:
        assert tuple(cnn.blocks[0].skip_projection.weight.shape) == (64, 128)
        assert cnn.blocks[0].skip_projection.weight.grad is not None
    print('Smoke test PASS: P=128, C=', channels)
del cnn


## Train và evaluate
Mỗi cấu hình dùng seed mô hình 234 và DataLoader seed 234. Batch 64 Normal + 64 abnormal, LR 1e-5, milestones 4/8. Không ghi đè checkpoint cũ.

In [ ]:
def run_command(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=SRC_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError('Tiến trình thất bại; traceback gốc đã in ở trên.')

results = {}
for channels in CHANNELS_TO_RUN:
    variant = f'k5_head_topk_p128_c{channels}'
    checkpoint_dir = SRC_DIR / 'model' / 'epoch_checkpoints_cnn_vadclip' / RUN_TAG / variant
    checkpoint = checkpoint_dir / f'epoch_{EPOCH:03d}.pth'
    train_dir = OUTPUT_ROOT / 'train' / RUN_TAG / variant
    eval_dir = OUTPUT_ROOT / 'evaluate' / RUN_TAG / variant
    analyze_dir = OUTPUT_ROOT / 'analyze' / RUN_TAG / variant
    config_path = train_dir / 'config.json'
    expected = {'projection_width': 128, 'cnn_channels': channels, 'kernel_size': 5,
                'score_mode': 'scoring_head', 'residual_mode': 'on', 'aggregation': 'topk',
                'lambda_alignment': 1.0, 'lambda_text': 1.0, 'seed': SEED,
                'data_seed': DATA_SEED, 'batch_size': 64, 'lr': 1e-5,
                'max_epoch': EPOCH, 'scheduler_milestones': [4, 8],
                'scheduler_rate': 0.1, 'visual_length': 256,
                'feature_root': str(FEATURE_ROOT),
                'train_list': str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')}
    if checkpoint.is_file():
        if not config_path.is_file():
            raise RuntimeError(f'{variant}: checkpoint có nhưng thiếu config; đổi RUN_TAG.')
        config = json.loads(config_path.read_text(encoding='utf-8'))
        if any(config.get(key) != value for key, value in expected.items()):
            raise RuntimeError(f'{variant}: checkpoint không khớp cấu hình; đổi RUN_TAG.')
        print('Bỏ qua train, checkpoint đã có:', checkpoint)
    else:
        train_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_train_cnn_vadclip.py'),
                     '--feature-root', str(FEATURE_ROOT), '--train-list', expected['train_list'],
                     '--projection-width', '128', '--cnn-channels', str(channels),
                     '--kernel-size', '5', '--score-mode', 'scoring_head',
                     '--residual-mode', 'on', '--aggregation', 'topk',
                     '--lambda-alignment', '1.0', '--lambda-text', '1.0',
                     '--seed', str(SEED), '--data-seed', str(DATA_SEED),
                     '--batch-size', '64', '--lr', '1e-5', '--max-epoch', str(EPOCH),
                     '--scheduler-milestones', '4', '8', '--scheduler-rate', '0.1',
                     '--num-workers', '2', '--visual-length', '256',
                     '--epoch-checkpoint-dir', str(checkpoint_dir),
                     '--output-model-path', str(SRC_DIR / 'model' / 'cnn_vadclip' / RUN_TAG / f'{variant}.pth'),
                     '--results-dir', str(train_dir)]
        run_command(train_cmd)
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    metrics_path = eval_dir / 'metrics.json'
    if metrics_path.is_file():
        old = json.loads(metrics_path.read_text(encoding='utf-8'))
        if old.get('model_path') != str(checkpoint) or old.get('cnn_channels') != channels:
            raise RuntimeError(f'{variant}: metrics không đúng checkpoint/cấu hình; đổi RUN_TAG.')
        print('Bỏ qua evaluate, đã có metrics:', metrics_path)
    else:
        eval_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_cnn_vadclip.py'),
                    '--model-type', 'cnn', '--model-path', str(checkpoint),
                    '--feature-root', str(FEATURE_ROOT),
                    '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
                    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
                    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
                    '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
                    '--projection-width', '128', '--cnn-channels', str(channels),
                    '--kernel-size', '5', '--score-mode', 'scoring_head',
                    '--residual-mode', 'on', '--train-config-json', str(config_path),
                    '--evaluate-dir', str(eval_dir), '--analyze-dir', str(analyze_dir)]
        run_command(eval_cmd)
    results[channels] = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Checkpoint:', checkpoint)
    print('Train:', train_dir, 'Evaluate:', eval_dir, 'Analyze:', analyze_dir)

keys = ('classifier_auc', 'classifier_ap', 'alignment_auc', 'alignment_ap', 'ano_auc', 'avg_mAP')
print('Model | AUC1 | AP1 | AUC2 | AP2 | Ano-AUC | mAP')
for channels in CHANNELS_TO_RUN:
    print(f'P128/C{channels} | ' + ' | '.join(f'{results[channels][key]:.2f}' for key in keys))
if 128 in results and 64 in results:
    print('Delta C64-C128 | ' + ' | '.join(f'{results[64][key]-results[128][key]:+.2f}' for key in keys))
    for channels in CHANNELS_TO_RUN:
        path = OUTPUT_ROOT / 'analyze' / RUN_TAG / f'k5_head_topk_p128_c{channels}' / 'summary.json'
        summary = json.loads(path.read_text(encoding='utf-8'))
        print(f'C{channels} Top-15 anomaly fraction (macro): {summary["selected_anomaly_fraction_abnormal_video_macro"]:.3f}')
